# INFO 490 | Assignment 7 (Part 2), A7_P2_S2: RAG With Our Small Language Model

**Question:** If we give the SLM retrieved documents at inference time, without changing its weights, do its answers improve?

**Pipeline (class Week 6 RAG notebook):** knowledge base → TF-IDF vectors → cosine-similarity retrieval → add retrieved text
to the input → generate with the **same SLM selected in Part 1** (`tiny_transformer.pt` from `A7_P2_S1.ipynb`, Experiment 2: 48 / 2 / 1 / 192).

**Compatibility adjustments for the SLM** (allowed by the assignment):
1. **Tokenizer:** the SLM is character-level with only 27 characters, so all text it reads is lowercase, without digits or "?".
2. **Context window:** the SLM can only see the **last 32 characters** of its input (class `generate_text` keeps
   `ids[:, -context_length:]`). We keep the class prompt order (documents first, then the question) and record exactly
   which characters the SLM could see.
3. **No instructions:** the class RAG prompt has instructions ("Answer the QUESTION using only the DOCUMENTS...").
   Our SLM was never trained to follow instructions, and they would not fit in 32 characters, so we drop them.

In [1]:
# ============================================================
# IMPORTS, SEED, AND SHARED SETTINGS
# ============================================================

import time
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
DEVICE = torch.device("cpu")

pd.set_option("display.max_colwidth", None)

# Generation settings used for EVERY condition in Parts 2-4
# 80 characters: long enough for the longest manual sentence (66)
MAX_NEW_TOKENS = 80
TEMPERATURE = 0.5

print("PyTorch:", torch.__version__, "| Device:", DEVICE)


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PyTorch: 2.2.2 | Device: cpu


In [2]:
# ============================================================
# CLASS SLM CODE (copied unchanged from the Week 7 notebook)
# ============================================================
# TransformerBlock, TinyTransformerLM, generate_text, and
# calculate_loss are the class versions. We need them here to
# rebuild the model from the saved checkpoint.
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, feedforward_dim):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=0.0,
            batch_first=True)

        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

        self.feedforward = nn.Sequential(
            nn.Linear(embedding_dim, feedforward_dim),
            nn.ReLU(),
            nn.Linear(feedforward_dim, embedding_dim))

    def forward(self, x):
        T = x.shape[1]
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        normalized = self.ln1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)
        x = x + attended
        x = x + self.feedforward(self.ln2(x))
        return x


class TinyTransformerLM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, feedforward_dim, context_length):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embedding_dim, num_heads, feedforward_dim)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids):
        B, T = input_ids.shape

        if T > self.context_length:
            raise ValueError("Input sequence exceeds context length.")

        positions = torch.arange(T, device=input_ids.device)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        x = self.ln_final(x)

        return self.output(x)


@torch.no_grad()
def generate_text(model, prompt, max_new_tokens=100, temperature=1.0):
    if temperature <= 0:
        raise ValueError("Temperature must be greater than zero.")

    was_training = model.training
    model.eval()

    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)

    for _ in range(max_new_tokens):
        # Keep only the most recent context_length characters.
        context = ids[:, -model.context_length:]
        logits = model(context)
        next_logits = logits[:, -1, :] / temperature
        probabilities = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probabilities, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)

    if was_training:
        model.train()
    return decode(ids[0].tolist())


def calculate_loss(logits, targets):
    B, T, V = logits.shape
    return F.cross_entropy(logits.reshape(B * T, V), targets.reshape(B * T))

In [3]:
# ============================================================
# LOAD THE CHECKPOINT SELECTED IN PART 1
# ============================================================
# tiny_transformer.pt is the checkpoint saved by class Cell 14D
# in our Part 1 notebook (A7_P2_S1.ipynb, Experiment 2). Parts 2,
# 3, and 4 all load THIS file. Its vocabulary rebuilds the class
# character tokenizer (stoi / itos).
# Never commit .pt files (they are in .gitignore).
# ============================================================

CHECKPOINT_PATH = Path("tiny_transformer.pt")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

chars = checkpoint["vocab"]
config = checkpoint["config"]
CONTEXT_LENGTH = config["context_length"]

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}


def encode(s):
    return [stoi[ch] for ch in s]


def decode(ids):
    return "".join(itos[int(i)] for i in ids)


def load_selected_model():
    """A fresh copy of the selected Part 1 model (never shared between conditions)."""
    m = TinyTransformerLM(vocab_size=len(chars), **config).to(DEVICE)
    m.load_state_dict(checkpoint["model_state"])
    m.eval()
    return m


model = load_selected_model()

print("Checkpoint:", CHECKPOINT_PATH)
print("Configuration:", config)
print("Vocabulary:", repr("".join(chars)))
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

Checkpoint: tiny_transformer.pt
Configuration: {'embedding_dim': 48, 'num_heads': 2, 'num_layers': 1, 'feedforward_dim': 192, 'context_length': 32}
Vocabulary: '\n .abcdefghijklmnoprstuvwxy'
Parameters: 32,523


# Part 2: Retrieval-Augmented Generation Using Our SLM

## Step 2.1: Create a New Knowledge Base

Our knowledge base is a short **SafeSpace manual** (`safespace_manual.txt`), a user guide for our team's wellness app.
The notebook splits it into **8 sections**, one document each (D1–D8). D8 is a **distractor** about a different app, so we
can see whether retrieval picks the wrong document.

In [4]:
# ============================================================
# STEP 2.1: OUR NEW KNOWLEDGE BASE: THE SAFESPACE MANUAL
# ============================================================
# safespace_manual.txt is a short user manual for SafeSpace,
# our team's wellness app. None of it appears in
# training_corpus.txt.
#
# CHUNKING: the manual is split at each "## ID | Title" line,
# so every section becomes one document with a unique ID.
#
# TOKENIZER COMPATIBILITY:
# The SLM only knows 27 characters: lowercase a-y (no q or z),
# space, period, and newline. So the text the SLM reads has
# no digits, capitals, "?", or apostrophes. Numbers are words
# ("nine eight eight" = the 988 crisis line), questions end with
# a period, and the app's "quests" are called "tasks" because
# the SLM cannot write the letter q.
# The title is only used by the retriever, never shown to the SLM.
# ============================================================

MANUAL_PATH = Path("safespace_manual.txt")

documents = []
for section in MANUAL_PATH.read_text(encoding="utf-8").split("## ")[1:]:
    header, body = section.split("\n", 1)
    doc_id, title = [part.strip() for part in header.split("|")]
    documents.append({"id": doc_id, "title": title, "text": " ".join(body.split())})

# The five evaluation questions (used in Parts 2, 3, and 4).
# Each must fit in the 32-character window with its newline.
EVAL = [
    {"id": "Q1", "question": "who can read my journal.", "expected": "safespace journals are private. only you can read your journal.",
     "facts": ["private", "only you"], "topic": ["journal", "private", "read"], "source": "D1"},
    {"id": "Q2", "question": "what is my task reward.", "expected": "each completed task unlocks a reward like a sticker or a frame.",
     "facts": ["sticker", "frame"], "topic": ["reward", "sticker", "frame", "task"], "source": "D2"},
    {"id": "Q3", "question": "how do i post in a forum.", "expected": "you must join a forum before you can post or reply.",
     "facts": ["join"], "topic": ["forum", "post", "reply"], "source": "D3"},
    {"id": "Q4", "question": "how does a joint task work.", "expected": "a joint task counts for both friends after the invite is accepted.",
     "facts": ["both friends", "accepted"], "topic": ["joint", "friend", "invite", "task"], "source": "D4"},
    {"id": "Q5", "question": "i feel unsafe.", "expected": "if you feel unsafe call or text nine eight eight.",
     "facts": ["nine eight eight"], "topic": ["call", "text", "unsafe", "help"], "source": "D6"},
]


def check_vocab(s):
    """Raise an error if the SLM's tokenizer cannot represent s."""
    missing = sorted(set(s) - set(chars))
    if missing:
        raise ValueError(f"Characters not in the SLM vocabulary: {missing} in {s!r}")
    return s


for d in documents:
    check_vocab(d["text"])
for q in EVAL:
    check_vocab(q["question"] + "\n" + q["expected"])
    assert len(q["question"]) + 1 <= CONTEXT_LENGTH, q["question"]

print("All", len(documents), "documents and", len(EVAL), "questions fit the SLM vocabulary.")
display(pd.DataFrame(documents))
display(pd.DataFrame(EVAL)[["id", "question", "expected", "source"]])

All 8 documents and 5 questions fit the SLM vocabulary.


,id,title,text
0,D1,Private Journals,safespace journals are private. only you can read your journal.
1,D2,Task Rewards,each completed task unlocks a reward like a sticker or a frame.
2,D3,Forums,you must join a forum before you can post or reply.
3,D4,Joint Tasks,a joint task counts for both friends after the invite is accepted.
4,D5,Daily Limit,each task can be completed once per day.
5,D6,Crisis Help,if you feel unsafe call or text nine eight eight.
6,D7,Friends,search for a username and send an invite to add a friend.
7,D8,Other App,studybuddy tutoring costs twelve dollars per session.


,id,question,expected,source
0,Q1,who can read my journal.,safespace journals are private. only you can read your journal.,D1
1,Q2,what is my task reward.,each completed task unlocks a reward like a sticker or a frame.,D2
2,Q3,how do i post in a forum.,you must join a forum before you can post or reply.,D3
3,Q4,how does a joint task work.,a joint task counts for both friends after the invite is accepted.,D4
4,Q5,i feel unsafe.,if you feel unsafe call or text nine eight eight.,D6


In [5]:
# ============================================================
# ANSWERING AND SCORING HELPERS
# ============================================================
# ask_slm():
#   The prompt is "<question>\n". The SLM continues the text;
#   the first generated line is its answer. A fixed seed makes
#   every condition use the same random draws.
#
# score_response(): Response Quality (1-10), following Step 2.3:
#   1 point   base
#   0-6       factual accuracy: share of the expected key facts present
#   +2        relevance: mentions the question's topic
#   +1        supported: no invented facts (no prices/times that are
#             not in the knowledge base answer) and not empty
# Key facts and topic words are listed with each question in EVAL.
# ============================================================

NUMBER_WORDS = {"one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve"}


def ask_slm(m, prompt, seed=SEED):
    torch.manual_seed(seed)
    output = generate_text(m, prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE)
    continuation = output[len(prompt):]
    answer = continuation.lstrip("\n").split("\n")[0]
    return answer


def score_response(answer, q):
    words = answer.replace(".", " ").split()
    facts_found = sum(f in answer for f in q["facts"])
    accuracy = 6 * facts_found / len(q["facts"])
    relevant = any(t in answer for t in q["topic"])
    supported = bool(words) and not invented_info(answer, q)
    return round(1 + accuracy + 2 * relevant + 1 * supported, 1)


def invented_info(answer, q):
    """Number words in the answer that are NOT in the expected answer (unsupported facts)."""
    words = set(answer.replace(".", " ").split())
    return sorted((words & NUMBER_WORDS) - set(q["expected"].split()))

## Step 2.2: Test the SLM Without RAG (baseline)

The input is only the question. This is **Condition A (Baseline)** in Part 4.

In [6]:
# ============================================================
# STEP 2.2: BASELINE WITHOUT RETRIEVAL
# ============================================================

baseline_rows = []

for q in EVAL:
    answer = ask_slm(model, q["question"] + "\n")
    baseline_rows.append({
        "Question": q["id"],
        "Question Text": q["question"],
        "SLM Response Without RAG": answer,
        "Response Quality (1-10)": score_response(answer, q),
    })

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv("part2_baseline_results.csv", index=False)
display(baseline_df)

,Question,Question Text,SLM Response Without RAG,Response Quality (1-10)
0,Q1,who can read my journal.,django applications use models and views.,2.0
1,Q2,what is my task reward.,inference uses previously learned parameters.,2.0
2,Q3,how do i post in a forum.,students develop artificial intelligence.,2.0
3,Q4,how does a joint task work.,inference uses previously learned parameters.,2.0
4,Q5,i feel unsafe.,students learn python programming.,2.0


## Step 2.3: Apply RAG to the Same SLM

**Retrieval** is the class TF-IDF + cosine similarity retriever (`stop_words="english"`, `ngram_range=(1, 2)`,
title + text, `min_score=0.01`).

**Augmentation:** the SLM input is the retrieved document texts (most relevant first, as in the class prompt), then the
question, then a newline. The SLM then continues the text.

In [7]:
# ============================================================
# STEP 2.3A: BUILD THE TF-IDF INDEX AND RETRIEVER (CLASS VERSION)
# ============================================================

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
document_matrix = vectorizer.fit_transform([d["title"] + " " + d["text"] for d in documents])


def retrieve(query, top_k, min_score=0.01):
    """Return top matching document dictionaries with scores."""
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_matrix)[0]
    indices = scores.argsort()[::-1][:top_k]
    return [{**documents[i], "score": float(scores[i])} for i in indices if scores[i] >= min_score]


def build_slm_rag_input(question, matches):
    """Retrieved evidence, then the question. Plain text the SLM's tokenizer can read."""
    evidence = "\n".join(d["text"] for d in matches)
    return check_vocab(evidence + "\n" + question + "\n")


print("Document matrix shape:", document_matrix.shape)

# Retrieval accuracy on its own (did the correct source document come first?)
for q in EVAL:
    top = retrieve(q["question"], top_k=5)
    print(f"{q['id']} expected {q['source']} | retrieved {[(d['id'], round(d['score'], 2)) for d in top]}")

Document matrix shape: (8, 87)
Q1 expected D1 | retrieved [('D1', 0.42)]
Q2 expected D2 | retrieved [('D2', 0.4), ('D5', 0.15), ('D4', 0.11)]
Q3 expected D3 | retrieved [('D3', 0.47)]
Q4 expected D4 | retrieved [('D4', 0.56), ('D2', 0.16), ('D5', 0.11)]
Q5 expected D6 | retrieved [('D6', 0.58)]


In [8]:
# ============================================================
# STEP 2.3B: RUN RAG WITH TOP-1, TOP-2, TOP-3, AND TOP-5
# ============================================================
# "Visible to SLM" = the last CONTEXT_LENGTH characters of the
# input: the only part the model can actually read.
# "Evidence Chars Visible" = how many of those characters come
# from retrieved documents rather than the question.
# The model's weights are checked before and after.
# ============================================================

weights_before_rag = {k: v.clone() for k, v in model.state_dict().items()}

rag_rows = []

for q in EVAL:
    for top_k in [1, 2, 3, 5]:
        matches = retrieve(q["question"], top_k=top_k)
        slm_input = build_slm_rag_input(q["question"], matches)
        answer = ask_slm(model, slm_input)

        question_part = len(q["question"]) + 1
        evidence_visible = max(0, CONTEXT_LENGTH - question_part)

        rag_rows.append({
            "Question": q["id"],
            "Top-K": top_k,
            "Retrieved Documents": ", ".join(f"{d['id']} ({d['score']:.2f})" for d in matches),
            "Correct Doc Retrieved": q["source"] in [d["id"] for d in matches],
            "Input Length (chars)": len(slm_input),
            "Visible to SLM": slm_input[-CONTEXT_LENGTH:],
            "Evidence Chars Visible": evidence_visible,
            "Generated Response": answer,
            "Response Quality (1-10)": score_response(answer, q),
        })

rag_df = pd.DataFrame(rag_rows)
rag_df.to_csv("part2_rag_results.csv", index=False)
display(rag_df)

unchanged = all(torch.equal(weights_before_rag[k], v) for k, v in model.state_dict().items())
print("RAG changed model weights:", not unchanged)

,Question,Top-K,Retrieved Documents,Correct Doc Retrieved,Input Length (chars),Visible to SLM,Evidence Chars Visible,Generated Response,Response Quality (1-10)
0,Q1,1,D1 (0.42),True,89,urnal.\nwho can read my journal.\n,7,django applications use models and views.,2.0
1,Q1,2,D1 (0.42),True,89,urnal.\nwho can read my journal.\n,7,django applications use models and views.,2.0
2,Q1,3,D1 (0.42),True,89,urnal.\nwho can read my journal.\n,7,django applications use models and views.,2.0
3,Q1,5,D1 (0.42),True,89,urnal.\nwho can read my journal.\n,7,django applications use models and views.,2.0
4,Q2,1,D2 (0.40),True,88,frame.\nwhat is my task reward.\n,8,the profes artificial intelligence.,2.0
5,Q2,2,"D2 (0.40), D5 (0.15)",True,129,er day.\nwhat is my task reward.\n,8,the profes artificial intelligence.,2.0
6,Q2,3,"D2 (0.40), D5 (0.15), D4 (0.11)",True,196,cepted.\nwhat is my task reward.\n,8,the profes artificial intelligence.,2.0
7,Q2,5,"D2 (0.40), D5 (0.15), D4 (0.11)",True,196,cepted.\nwhat is my task reward.\n,8,the profes artificial intelligence.,2.0
8,Q3,1,D3 (0.47),True,78,eply.\nhow do i post in a forum.\n,6,students develop artificial intelligence.,2.0
9,Q3,2,D3 (0.47),True,78,eply.\nhow do i post in a forum.\n,6,students develop artificial intelligence.,2.0


RAG changed model weights: False


In [9]:
# ============================================================
# STEP 2.3C: SUMMARY BY RETRIEVAL CONFIGURATION
# ============================================================

summary = rag_df.groupby("Top-K").agg(
    avg_quality=("Response Quality (1-10)", "mean"),
    correct_doc_retrieved=("Correct Doc Retrieved", "sum"),
    avg_input_chars=("Input Length (chars)", "mean"),
).round(2)

summary.loc["No RAG"] = [baseline_df["Response Quality (1-10)"].mean().round(2), 0, None]
display(summary)

,avg_quality,correct_doc_retrieved,avg_input_chars
Top-K,,,
1,2.0,5.0,83.0
2,2.0,5.0,104.0
3,2.0,5.0,125.6
5,2.0,5.0,125.6
No RAG,2.0,0.0,NaN


### Extra check: does the SLM use evidence it CAN see?

In the runs above, the question fills most of the 32-character window. As a diagnostic (not one of the required
configurations), we drop the question and give the SLM only the **start of the correct document**, so the evidence is the
only thing in its window. If the SLM could use retrieved text, it should continue the SafeSpace sentence.

In [10]:
# ============================================================
# DIAGNOSTIC: EVIDENCE INSIDE THE WINDOW
# ============================================================
# Give the first half of the correct document as the prompt and
# see whether the SLM completes it with the real fact.
# ============================================================

diagnostic_rows = []

for q in EVAL:
    doc_text = next(d["text"] for d in documents if d["id"] == q["source"])
    words = doc_text.split()
    prompt = " ".join(words[: len(words) // 2]) + " "
    torch.manual_seed(SEED)
    continuation = generate_text(model, prompt, max_new_tokens=40, temperature=TEMPERATURE)[len(prompt):]
    diagnostic_rows.append({
        "Question": q["id"],
        "Prompt (start of correct document)": prompt,
        "SLM Continuation": continuation.split("\n")[0],
        "Real Ending": doc_text[len(prompt):],
    })

display(pd.DataFrame(diagnostic_rows))

,Question,Prompt (start of correct document),SLM Continuation,Real Ending
0,Q1,safespace journals are private. only,learned parameters.,you can read your journal.
1,Q2,each completed task unlocks a reward,l language models.,like a sticker or a frame.
2,Q3,you must join a forum,odels and views.,before you can post or reply.
3,Q4,a joint task counts for both,programming.,friends after the invite is accepted.
4,Q5,if you feel unsafe call,computers.,or text nine eight eight.


In [11]:
# ============================================================
# SAVE INPUTS FOR PART 4 (A7_P2_S3)
# ============================================================
# Part 4 compares Baseline, RAG, and Fine-Tuning. We use the
# Top-K with the best average quality (ties: the smallest K),
# and also measure consistency: the same question asked with
# five different random seeds. Consistency = share of seeds that
# give the most common answer.
# ============================================================

best_k = int(summary.drop(index="No RAG")["avg_quality"].astype(float).idxmax())
print("Top-K used in Part 4:", best_k)


def consistency(m, prompt, seeds=(1, 2, 3, 4, 5)):
    answers = [ask_slm(m, prompt, seed=s) for s in seeds]
    most_common = max(set(answers), key=answers.count)
    return answers.count(most_common) / len(answers)


part4_rows = []

for q in EVAL:
    rag_row = rag_df[(rag_df["Question"] == q["id"]) & (rag_df["Top-K"] == best_k)].iloc[0]
    rag_input = build_slm_rag_input(q["question"], retrieve(q["question"], top_k=best_k))
    part4_rows.append({
        "Question": q["id"],
        "Baseline SLM": baseline_df.loc[baseline_df["Question"] == q["id"], "SLM Response Without RAG"].iloc[0],
        "Baseline Consistency": consistency(model, q["question"] + "\n"),
        "SLM + RAG": rag_row["Generated Response"],
        "RAG Consistency": consistency(model, rag_input),
        "RAG Top-K": best_k,
    })

part4_df = pd.DataFrame(part4_rows)
part4_df.to_csv("part4_from_rag_notebook.csv", index=False)
display(part4_df)

Top-K used in Part 4: 1


,Question,Baseline SLM,Baseline Consistency,SLM + RAG,RAG Consistency,RAG Top-K
0,Q1,django applications use models and views.,1.0,django applications use models and views.,1.0,1
1,Q2,inference uses previously learned parameters.,0.8,the profes artificial intelligence.,0.8,1
2,Q3,students develop artificial intelligence.,0.4,students develop artificial intelligence.,0.6,1
3,Q4,inference uses previously learned parameters.,1.0,inference uses previously learned parameters.,1.0,1
4,Q5,students learn python programming.,0.6,students are building useful software products.,1.0,1


## Step 2.4: Analysis

1. **Did retrieved information change the responses?** Barely. Q1, Q3, and Q4 gave exactly the same answer as without RAG; Q2 changed
   to garbled text ("the profes artificial intelligence.") and Q5 to a different training sentence
   ("students are building useful software products."). No answer mentioned SafeSpace.
2. **Did RAG improve factual accuracy?** No. The retriever found the correct document first for **5/5** questions, but the SLM's
   answers contained **0** of the key facts at every Top-K, and average quality stayed **2.0**, the same as without RAG.
3. **Did more documents always help?** No. Top-2, 3, and 5 made the input longer (83 → about 126 characters) but gave the same answers as
   Top-1, because extra documents only push text further outside the window (Top-5 returned at most 3 documents, because the rest scored 0).
4. **Context length:** the SLM only reads the last **32 characters**. The question and its newline used 15–28 of them, so only
   **4–17 characters of evidence** were visible (for Q1 just "urnal.\n"), and the sentence with the answer was never in view.
   Even when the window held **only** the start of the correct document (diagnostic cell), the SLM finished it with training-corpus
   words ("learned parameters.") instead of the real ending.
5. **Why a large pretrained model behaves differently:** a large LLM has a context of thousands of tokens, knows English, and has
   learned to read, follow, and copy information from its prompt. Our character-level SLM only learned to continue 21 memorized
   sentences, so retrieved text is just unfamiliar characters to it. RAG depends on the generator being able to use its context.